In [1]:
from xaikd import datasets, models, utils, constants, attributors

from torch.utils.data import random_split, DataLoader

from torchvision.datasets import CIFAR100, ImageNet
from torchmetrics.classification import BinaryAUROC

from torch.nn import functional as F

import pandas as pd
import torch
import numpy as np 
import numpy.typing as npt

from tqdm.notebook import tqdm
from torch import nn

from matplotlib import pyplot as plt

In [2]:
DEVICE = utils.get_device()

DEVICE

DATA_SIZE = 0.01


SELECTED_GROUP = "butterfly"

In [3]:
dataset = datasets.construct("imagenet")

In [4]:
def get_fine_labels(name):

    return datasets.imagenet.IMAGENET_SUPERCLASS_MAPPING[name]
    # if "class" in name:
    #     label = int(name.replace("class", ""))
    #     return [label]
    # df = pd.read_csv(constants.CIFAR100_SUPER_CLASS_MAPPING)
    # return sorted(df[df.coarse_label_name == name].fine_label.values)
SELECTED_CLASSES = get_fine_labels(SELECTED_GROUP)
SELECTED_CLASSES

[321, 322, 323, 324, 325, 326]

In [5]:
NOT_SELECTED_CLASSES = sorted(set(np.arange(1000)).difference(SELECTED_CLASSES))

In [6]:
NOT_SELECTED_CLASSES

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [7]:
def target_transform(label):
    if label in SELECTED_CLASSES:
        return 1
    else:
        return 0

In [8]:
trng = torch.Generator()
trng.manual_seed(1)

transform = datasets.construct("imagenet").input_transformation
ds_train = ImageNet(root="../../datasets/imagenet", split="train", transform=transform, target_transform=target_transform)
ds_train, _ = random_split(ds_train, [DATA_SIZE, 1-DATA_SIZE])

ds_val = ImageNet(root="../../datasets/imagenet", split="val", transform=transform, target_transform=target_transform)

dl_train = DataLoader(ds_train, num_workers=12, shuffle=False, batch_size=128)
dl_val = DataLoader(ds_val, num_workers=12, shuffle=False,  batch_size=128)

In [9]:
def ano():
    for label, dl in [
        ("train", dl_train),
        ("val", dl_val),
    ]:
        arr_values = []
        for _, y in tqdm(dl):
            arr_values.extend(y.tolist())
        print(f"[subset={label}] p(y=1)={np.mean(arr_values):.4f}")
ano()

  0%|          | 0/101 [00:00<?, ?it/s]

[subset=train] p(y=1)=0.0058


  0%|          | 0/391 [00:00<?, ?it/s]

[subset=val] p(y=1)=0.0060


In [10]:
class LastLayerBinary(nn.Module):
    def __init__(self, fc):
        super().__init__()
        self.fc = fc

    def forward(self, x):
        logits = self.fc(x)
        lse_pos = torch.logsumexp(logits[:, SELECTED_CLASSES], dim=1)
        lse_neg = torch.logsumexp(logits[:, NOT_SELECTED_CLASSES], dim=1)
        return lse_pos - lse_neg
        
        
        # max_logit_selected = logits[:, SELECTED_CLASSES].max(dim=1).values
        # max_logit_notselected = logits[:, NOT_SELECTED_CLASSES].max(dim=1).values

        # assert len(max_logit_selected.shape) == 1

        # return max_logit_selected - 0* max_logit_notselected
        
    
def construct_model():
    model = models.get_trained_model("imagenet-resnet18-tv")

    model.fc = LastLayerBinary(model.fc)
    model.eval()
    model.to(DEVICE)
    
    return model

def sanity_check():
    
    model  = construct_model()

    output = model(torch.randn((5, 3, 224, 224)).to(DEVICE))
    assert output.shape == (5,)
    print("[Sanity check passed]: Model is callable")
    
sanity_check()

[Sanity check passed]: Model is callable


In [11]:
model = construct_model()
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [12]:
def compute_performance(model, verbose=False):
    dl =  dl_val
    metric = BinaryAUROC(thresholds=20)
    model.to(DEVICE)

    for x, y in tqdm(dl, disable=not verbose):
        x = x.to(DEVICE)
        logit = model(x).cpu()
        
        metric.update(logit, y)

    metric = float(metric.compute())
    metric = np.max([metric, 1-metric])
    return metric
    
ref_performance = compute_performance(model, verbose=True)    
ref_performance

  0%|          | 0/391 [00:00<?, ?it/s]

0.9933035969734192

# Extract Activation

In [13]:
class VoidAttributor:

    def __enter__(self, **kwargs):
        pass

    def __exit__(self, type, value, tb):
        pass

def compute_logodd_winning(logits):

    return torch.sign(logits).detach() * logits

def compute_logits(logits):

    return logits

class OutputQuantity:
    def __call__(self, logits, target_logits):
        raise NotImplementedError()
    def __str__(self):z
        return self.__class__.__name__


class LogOddSquared(OutputQuantity):
    def __call__(self, logits, targets):
        return (torch.sign(logits).detach() * logits).pow(2)


class LogOddWinningClass(OutputQuantity):
    def __call__(self, logits, targets):
        return torch.sign(logits).detach() * logits


class LogOddPositiveClass(OutputQuantity):
    def __call__(self, logits, targets):
        return logits

        
class LogOddIfPositiveClass(OutputQuantity):
    def __call__(self, logits, targets):
        return (logits >= 0)*logits


class LogOddIfNegativeClass(OutputQuantity):
    def __call__(self, logits, targets):
        return (logits < 0)*logits


class LogOddTargetClass(OutputQuantity):
    def __call__(self, logits, targets):
        return torch.sign(2*targets-1) * logits
        

class BinaryCrossEntropyWinning(OutputQuantity):
    def __call__(self, logits, targets):
        winning_target = torch.sign(logits).detach()
        return - F.binary_cross_entropy_with_logits(logits, winning_target)



IndentationError: unexpected indent (2913825285.py, line 21)

In [ ]:
OUTPUT_QUANTITY = LogOddWinningClass()

def extract_activation_context_for_task(
    model: nn.Module,
    layer: str,
    data_loader=dl_train,
    output_quantity=OUTPUT_QUANTITY,
    seed=1,
    device=DEVICE,
    number_of_selected_spatial_locations=20,
    strict_mode=False,
    verbose=False
):

    arr_logodds = []
    arr_act = []
    arr_ctx = []

    rng = np.random.default_rng(seed=1)

    try:
        
        module, hook = utils.interceptor.attach_hook_intercept_layer_output(
            model, layer, should_retain_grad=True, detach_output=False
        )

        
        for batch in tqdm(data_loader, desc=f"[layer={layer}] extract act ctx (wrt {output_quantity})"):
            x, y = batch
            x = x.to(device)


            logits = model(x)
            
            quantities = output_quantity(logits, y)
            
            (quantities).sum().backward()
            act = utils.interceptor.get_output(module)

            assert act.grad is not None
            ctx = act.grad
            
            output_dimensions = act.shape[1:]

            assert ctx.shape == act.shape

            act = act.detach().cpu().numpy()
            ctx = ctx.detach().cpu().numpy()

            selected_act, selected_ctx = utils.subsample_tensors(
                act,
                ctx,
                num_locations=number_of_selected_spatial_locations,
                rng=rng,
            )
            

            arr_act.append(selected_act)
            arr_ctx.append(selected_ctx)

    finally:
        hook.remove()
        model.fc.task_id = None

    print(f"{layer}: output-dims={output_dimensions}")

    arr_act = np.vstack(arr_act)
    arr_ctx = np.vstack(arr_ctx)
 
    return arr_act, arr_ctx


# def extract_activation_context_for_task(
#     model: nn.Module,
#     layer: str,
#     data_loader=dl_train,
#     output_quantity=LogOddPositiveClass(),
#     seed=1,
#     device=DEVICE,
#     number_of_selected_spatial_locations=20,
#     strict_mode=False,
#     verbose=False
# ):
        
#     return attributors.extract_activation_context(
#         model=model,
#         layer=layer,
#         dataset=dataset,
#         data_loader=dl_train,
#         logit_modifier=LogOddWinningClass(),
#         rng=np.random.default_rng(seed=seed),
#         device=device,
#         number_of_selected_spatial_locations=20,
#         strict_mode=False,
#     )

    # arr_act_2, arr_neg_ctx = attributors.extract_activation_context(
    #     model=model,
    #     layer=layer,
    #     dataset=dataset,
    #     data_loader=dl_train,
    #     logit_modifier=LogOddIfNegativeClass(),
    #     rng=np.random.default_rng(seed=seed),
    #     device=device,
    #     number_of_selected_spatial_locations=20,
    #     strict_mode=False,
    # )

    # np.testing.assert_allclose(arr_act, arr_act_2)

    # arr_ctx = np.concatenate(
    #     [
    #         arr_pos_ctx[:, np.newaxis], 
    #         arr_neg_ctx[:, np.newaxis]
    #     ], axis=1
    # )
    # # print(arr_ctx.shape)
    # # raise
    # return arr_act, arr_ctx
    

def ano():

    arr_act, arr_ctx = extract_activation_context_for_task(
        model, 
        "layer3",
        dl_train,
    )

    print(f"Sanity check: passed!")
ano()

# Construct Basis

In [ ]:
def _solve_eigvecs(cov, sort_func=lambda x: x):
    eigvals, eigvecs = np.linalg.eigh(cov)

    assert len(eigvals.shape) == 1

    indices = np.argsort(-sort_func(eigvals))
    eigvals = eigvals[indices]
    eigvecs = eigvecs[:, indices]

    return eigvecs, sort_func(eigvals)


class BasisInterface:
    def get_Uk(self, k: int) -> npt.NDArray:
        raise NotImplementedError()   

class PCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_act.T @ arr_act
        self.U, _ = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class GradPCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        cov = arr_ctx.T @ arr_ctx
        self.U, _ = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]
        
class PRCASortAbs(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):

        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U, _ = _solve_eigvecs(cov, sort_func=np.abs)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class PRCA(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):

        cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        self.U, _ = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

In [ ]:
# def construct_fh(Uk):
#     def fh(mod, inp, outp):
#         return F.conv2d(
#             outp,
#             (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
#         )
#     return fh
    
# class PRCASortAbsSF_L4K5(BasisInterface):
#     def __init__(
#         self, 
#         arr_act, 
#         arr_ctx, 
#         layer=None, 
#         arr_filtering_layers=[("layer4", 5)], 
#         task_id=None
#     ):
#         arr_hooks = []
#         self.pre_filters = []

#         try:
                
#             for filter_layer, k_layer in arr_filtering_layers:
#                 print(f"Filtering at {filter_layer}@k{k_layer}")
#                 basis_layer = GradPCASF_L3K20L4K5(None, None, layer=filter_layer, arr_filtering_layers=[])
#                 module = getattr(model, filter_layer)
#                 Uk = basis_layer.get_Uk(k=k_layer)
#                 hook = module.register_forward_hook(construct_fh(torch.from_numpy(Uk).float().to(DEVICE)))
#                 arr_hooks.append(hook)
                
#                 self.pre_filters.append((filter_layer, Uk))
                         
#             arr_act, arr_ctx = extract_activation_context_for_task(
#                 model, 
#                 layer,
#                 dl_train,
#             )
    
#             cov = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
#             self.U, self.eigvals = _solve_eigvecs(cov, sort_func=np.abs)
#         finally:
#             for hook in arr_hooks:
#                 hook.remove()
   
#     def get_Uk(self, k: int):
#         return self.U[:, :k]

# class PRCASortAbsSF_L3K20L4K5(PRCASortAbsSF_L4K5):
#     def __init__(
#         self, 
#         arr_act, 
#         arr_ctx, 
#         layer=None, 
#         arr_filtering_layers=[
#             ("layer4", 5),
#             ("layer3", 20),
#         ], 
#         task_id=None
#     ):
#         super().__init__(
#             arr_act=arr_act,
#             arr_ctx=arr_ctx,
#             layer=layer,
#             arr_filtering_layers=arr_filtering_layers
#         )
# class PRCASortAbsSF_L3K20(PRCASortAbsSF_L4K5):
#     def __init__(
#         self, 
#         arr_act, 
#         arr_ctx, 
#         layer=None, 
#         arr_filtering_layers=[
#             ("layer3", 50),
#         ], 
#         task_id=None
#     ):
#         super().__init__(
#             arr_act=arr_act,
#             arr_ctx=arr_ctx,
#             layer=layer,
#             arr_filtering_layers=arr_filtering_layers
#         )

# def ano():

#     layer = "layer2"
    

#     basis_1 = PRCASortAbsSF_L3K20(
#         arr_act=None,
#         arr_ctx=None,
#         layer=layer
#     )
#     basis_1.get_Uk(k=10)

#     plt.plot(basis_1.eigvals, label="w/ filtering")
    
#     basis_2 = PRCASortAbsSF_L3K20(
#         arr_act=None,
#         arr_ctx=None,
#         layer=layer,
#         arr_filtering_layers=[]
#     )
#     basis_2.get_Uk(k=10)
#     plt.plot(basis_2.eigvals, label="w/o filtering")
#     print(f"Sanity check: passed!")
#     plt.legend()
# ano()

In [ ]:
    
# class GradPCASF_L4K5(BasisInterface):
#     def __init__(
#         self, 
#         arr_act, 
#         arr_ctx, 
#         layer=None, 
#         arr_filtering_layers=[("layer4", 5)], 
#         task_id=None
#     ):
#         arr_hooks = []

#         self.pre_filters = []
#         try:
                
#             for filter_layer, k_layer in arr_filtering_layers:
#                 print(f"Filtering at {filter_layer}@k{k_layer}")
#                 basis_layer = GradPCASF_L4K5(None, None, layer=filter_layer, arr_filtering_layers=[])
#                 module = getattr(model, filter_layer)
#                 raw_Uk = basis_layer.get_Uk(k=k_layer)
#                 Uk = torch.from_numpy(raw_Uk).float().to(DEVICE)
#                 hook = module.register_forward_hook(construct_fh(Uk))
#                 arr_hooks.append(hook)
                
#                 self.pre_filters.append((filter_layer, raw_Uk))
                         
#             arr_act, arr_ctx = extract_activation_context_for_task(
#                 model, 
#                 layer,
#                 dl_train,
#             )
    
#             cov = arr_ctx.T @ arr_ctx
#             self.U, self.eigvals = _solve_eigvecs(cov)
#         finally:
#             for hook in arr_hooks:
#                 hook.remove()
   
#     def get_Uk(self, k: int):
#         return self.U[:, :k]

# class GradPCASF_L3K20L4K5(GradPCASF_L4K5):
#     def __init__(
#         self, 
#         arr_act, 
#         arr_ctx, 
#         layer=None, 
#         arr_filtering_layers=[
#             ("layer4", 5),
#             ("layer3", 20),
#         ], 
#         task_id=None
#     ):
#         super().__init__(
#             arr_act=arr_act,
#             arr_ctx=arr_ctx,
#             layer=layer,
#             arr_filtering_layers=arr_filtering_layers
#         )

# def ano():

#     layer = "layer2"
    

#     basis_1 = GradPCASF_L3K20L4K5(
#         arr_act=None,
#         arr_ctx=None,
#         layer=layer
#     )
#     basis_1.get_Uk(k=10)

#     plt.plot(basis_1.eigvals, label="w/ filtering")
    
#     basis_2 = GradPCASF_L3K20L4K5(
#         arr_act=None,
#         arr_ctx=None,
#         layer=layer,
#         arr_filtering_layers=[]
#     )
#     basis_2.get_Uk(k=10)
#     plt.plot(basis_2.eigvals, label="w/o filtering")
#     print(f"Sanity check: passed!")
#     plt.legend()
# ano()

In [ ]:
class PRCAReconNonGreedyLearner:
  
    def fit(
        self, arr_act: npt.NDArray, arr_ctx: npt.NDArray, k: int,  U_init=None, device="cpu",
    ) -> npt.NDArray:
        n, d = arr_act.shape

        # assert arr_ctx.shape == arr_act.shape, arr_ctx.shape
        
        lr = 1e-4
        epochs = 5000
                
        n, d,  = arr_act.shape

        arr_act = arr_act / ((np.mean(arr_act**2) ** (1 / 2)) * (d ** (1 / 4)))
        arr_ctx = arr_ctx / ((np.mean(arr_ctx**2) ** (1 / 2)) * (d ** (1 / 4)))
        
        arr_act: torch.Tensor = torch.from_numpy(arr_act).to(device)
        arr_ctx: torch.Tensor = torch.from_numpy(arr_ctx).to(device)


        linear_layer = torch.nn.Linear(k, d, bias=False)
        trng = torch.Generator()
        trng.manual_seed(1)
        if U_init is None:
            
            U_init = torch.randn((k, d), generator=trng)
        else:
            U_init = torch.from_numpy(U_init.T)
            
        linear_layer.weight = torch.nn.Parameter(U_init)
    
        ortho_layer = torch.nn.utils.parametrizations.orthogonal(linear_layer).to(device)
        assert ortho_layer.weight.shape == (k, d)
        
        optimizer = torch.optim.Adam(ortho_layer.parameters(), lr=lr)

        pgb = tqdm(range(epochs), desc=f"{self.__class__.__name__} (k={k})")
        for epoch in pgb:
            optimizer.zero_grad()
            
            # shape: (k, d)
            U = ortho_layer.weight

            act_proj = arr_act @ U.T @ U

            residue = act_proj - arr_act

            ctx_residue = (arr_ctx * residue).sum(dim=1)
            
            # assert ctx_residue.shape == (n, 2), ctx_residue.shape
            loss = ctx_residue.pow(2)
            
            loss = loss.mean()
            
            loss.backward()
        
            optimizer.step()

            loss = loss.detach().cpu().numpy()
                
            pgb.set_description_str(f"{self.__class__.__name__} (k={k}) loss={loss:.4e}")
            
        U =  ortho_layer.weight.T.detach().cpu().numpy()
        
        # sanity_check
        np.testing.assert_allclose(U.T @ U, np.eye(k), atol=1e-4)

        return U


class PRCARecon(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):
        self.arr_act = arr_act
        self.arr_ctx = arr_ctx
      
        self.layer = layer
        self.task_id = task_id
        self.is_slow = True
        
    def get_Uk(self, k: int):

        return PRCAReconNonGreedyLearner(
            
        ).fit(
            self.arr_act, self.arr_ctx, 
            k=k,
            U_init=PCA(
                arr_act=self.arr_act,
                arr_ctx=self.arr_ctx
            ).get_Uk(k),
            device=DEVICE
        )

In [ ]:
class SumSigmaASigmaACNormalizedBefore(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):

        pca = PCA(arr_act=arr_act, arr_ctx=arr_ctx)
        gradpca = GradPCA(arr_act=arr_act, arr_ctx=arr_ctx)
        
        arr_act = arr_act / (pca.eigvals[0] ** 0.5)
        arr_ctx = arr_ctx / (gradpca.eigvals[0] ** 0.5)


        cov_a = 2*(arr_act.T @ arr_act)
        
        cov_ac = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        cov = cov_ac + cov_a 
        
        self.U, self.eigvals = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]

class SumSigmaASigmaCSigmaACNormalizedBefore(BasisInterface):
    def __init__(self, arr_act, arr_ctx, layer=None, task_id=None):

        pca = PCA(arr_act=arr_act, arr_ctx=arr_ctx)
        gradpca = GradPCA(arr_act=arr_act, arr_ctx=arr_ctx)
        
        arr_act = arr_act / ( np.sum(pca.eigvals) ** 0.5)
        arr_ctx = arr_ctx / ( np.sum(gradpca.eigvals) ** 0.5)


        cov_a = 2*(arr_act.T @ arr_act)
        cov_c = 2*(arr_ctx.T @ arr_ctx)
        cov_ac = arr_act.T @ arr_ctx + arr_ctx.T @ arr_act
        
        cov = cov_ac + cov_a + cov_c
        
        self.U, self.eigvals = _solve_eigvecs(cov)
        
    def get_Uk(self, k: int):
        return self.U[:, :k]


# Estimate Performance

In [ ]:
def construct_fh(Uk):
    def fh(mod, inp, outp):
        return F.conv2d(
            outp,
            (Uk@Uk.T).unsqueeze(2).unsqueeze(3)
        )
    return fh

def compute_task_aurocs_at_k(
    model, layer, 
    arr_ks,
    arr_basis_names=[
        PCA
    ],
    
):
    arr_act, arr_ctx = extract_activation_context_for_task(
        model=model, 
        layer=layer, 

    )
    
    module = getattr(model, layer)

    arr_dfs = []
    for basis_class in arr_basis_names:
        arr_stat_rows = []

        basis: BasisInterface = basis_class(
            arr_act=arr_act, 
            arr_ctx=arr_ctx,
            layer=layer,
        )
        basis_name = basis_class.__name__

        for k in tqdm(
            arr_ks,
            desc=f"[{basis_name:>20s}] Estimating Performance"
        ):
        
            if ("-k" in basis_name) and not f"{k}" == basis_name.split("k")[1]:
                continue

            Uk = basis.get_Uk(k=k)

            row = dict(
                data_group=SELECTED_GROUP,
                layer=layer,
                k=k,
                basis_name=basis_name,
            )

            Uk = torch.from_numpy(Uk).float().to(DEVICE)

            arr_hooks = []
            try:
                hook = module.register_forward_hook(construct_fh(Uk))
                arr_hooks.append(hook)

                if hasattr(basis, "pre_filters") and len(basis.pre_filters) > 0:
                    print("we pre filter")
                    for filter_layer, filter_Uk in basis.pre_filters:
                        
                        layer_module = getattr(model, filter_layer)
                        hook = layer_module.register_forward_hook(
                            construct_fh(
                                torch.from_numpy(filter_Uk).float().to(DEVICE)
                            )
                        )
                        arr_hooks.append(hook)
                
                for label, dl in [
                    # ("train", dl_train),
                    ("val", dl_val)
                ]:
                    row[f"{label}_auroc"] = compute_performance(model)
                
            finally:
                for hook in arr_hooks:
                    hook.remove()

            if hasattr(basis, "is_slow"):
                print(f"k={k}", row)
            arr_stat_rows.append(row)
            
     
        df = pd.DataFrame(arr_stat_rows)

        arr_dfs.append(df)

    df = pd.concat(arr_dfs).sort_values(by=["k", f"val_auroc"], ascending=[True, False])
    
    return df

compute_task_aurocs_at_k(
    model, layer="layer2", 
    arr_basis_names=[
        PCA,
        GradPCA,
        PRCASortAbs,
        SumSigmaASigmaACNormalizedBefore,
        SumSigmaASigmcaCSigmaACNormalizedBefore,
        # PRCARecon,
    ],
    arr_ks=[
        # 1, 
        # 3, 
        20,
        30,
    ],
)

In [ ]:
raise

## Estimate Statistics

In [ ]:
LAYER_DIM_MAPPING = utils.get_dimensions_at_layers(model, dl_train, ["layer1", "layer2", "layer3", "layer4"], device=DEVICE)

In [ ]:
def estimate_stats(
    arr_basis_names=[
        PCA,
        GradPCA,
        # PRCA,     
        PRCASortAbs,
        # PRCASortAbsSF_L4K5,
        # PRCASortAbsSF_L3K20L4K5,
        # PRCARecon
    ]
):
    arr_dfs = []
    arr_layers = ["layer1", "layer2", "layer3", "layer4"]
    # arr_layers = ["layer2"]
    for layer in tqdm(arr_layers):
        d = LAYER_DIM_MAPPING[layer]
        arr_ks = np.linspace(1, 64, 10).astype(int)

        df = compute_task_aurocs_at_k(
            model, layer=layer, 
            arr_basis_names=arr_basis_names,
            arr_ks=arr_ks,
        )
        arr_dfs.append(df)
    return pd.concat(arr_dfs)
df_stats = estimate_stats()

In [ ]:
df_stats[df_stats.layer == "layer2"]

In [ ]:
def viz(df):

    def ls_color_mapping(basis_name):
        return dict(
            PCA=("-", "blue"),
            GradPCA=(":", "blue"),
            PRCA=(":", "red"),
            PRCASortAbs=("--", "red"),
            PRCASortAbsSequentialFiltering=("-", "red"),
        ).get(basis_name, (None, None))
    
    arr_layers = df.layer.unique()
    ncols = len(arr_layers)
    plt.figure(figsize=(4*ncols, 3))
    plt.suptitle(f"imagenet-resnet16-tv: Task `{SELECTED_GROUP}-vs-others` (DATA_SIZE={DATA_SIZE}, output={OUTPUT_QUANTITY})", y=1.05)

    for lix, layer in enumerate(arr_layers):
        plt.subplot(1, ncols, lix+1)
        plt.title(f"layer={layer}")
        d = LAYER_DIM_MAPPING[layer]
        
        for bix, basis_name in enumerate(df.basis_name.unique()):
            _df = df[(df.basis_name == basis_name) & (df.layer==layer)]
            if bix == 0:
                plt.axhline(ref_performance, label="Original", color="k", lw=1, ls="-")
            
            ls, color = ls_color_mapping(basis_name)
            plt.plot(
                _df.k,
                _df.val_auroc,
                marker=".",
                label=basis_name,
                ls=ls,
                color=color
            )

        if lix == 0:
            plt.ylabel("AUROC")
            plt.legend()
        plt.xlabel(f"Subspace Dimensions (d={d})")

viz(df_stats)